In [ ]:
import torchvision
import numpy as np
import pandas as pd
from random import randint
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from tqdm import tqdm
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from time import time
import sklearn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import PIL
import os
import cv2

import warnings
warnings.filterwarnings('ignore')

In [ ]:
using_colab = True
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything-2.git'

    !mkdir -p images
    !wget -P images https://raw.githubusercontent.com/facebookresearch/segment-anything-2/main/notebooks/images/cars.jpg

    !mkdir -p ../checkpoints/
    !wget -P ../checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

if device.type == "cuda":
    print("Using b16-float")
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    
device

In [ ]:
np.random.seed(3)

def show_anns(anns, borders=True):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:, :, 3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.5]])
        img[m] = color_mask 
        if borders:
            import cv2
            contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
            # Try to smooth contours
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0, 0, 1, 0.4), thickness=1) 

    ax.imshow(img)
    return img

In [ ]:
images_path = "/kaggle/input/vqa-rad/images/"
csv_path = "/kaggle/input/vqa-rad/vqa-rad.csv"
df = pd.read_csv(csv_path)

In [ ]:
image_files = df["image_name"].values

In [ ]:
width, new_height, height, aspect_ratio, width/aspect_ratio

In [ ]:
width, height = Image.open(images_path + image_files[20][:-4]).size
aspect_ratio = width/height
width = 256
new_height = int(width/aspect_ratio)
Image.open(images_path + image_files[1][:-4]).resize((width, new_height), Image.LANCZOS)

In [ ]:
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

sam2_checkpoint = "../checkpoints/sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"

sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device, apply_postprocessing=True)

mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2,
    points_per_side=16,
    points_per_batch=64,
    pred_iou_thresh=0.8,
    stability_score_thresh=0.7,
    stability_score_offset=1.3,
    crop_n_layers=1,
    box_nms_thresh=0.8,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=100.0,
    use_m2m=True,
    multimask_output=True
)

In [ ]:
def get_sam_results(image):
    sam_result = mask_generator.generate(image)
    return sam_result

In [ ]:
sam_results = []

for file in tqdm(range(len(image_files))):
    width, height = Image.open(images_path + image_files[file][:-4]).size
    aspect_ratio = width/height
    width = 256
    new_height = int(width/aspect_ratio)
    image_sample = np.array(Image.open(images_path + image_files[file][:-4]).resize((width, new_height), Image.LANCZOS))
    try:
        sam_result = get_sam_results(image_sample)
        vqa = df[df["image_name"] == image_files[file]].to_dict(orient='records')
        sam_results.append({
            "filename" : image_files[file][:-4],
            "image" : image_sample,
            "vqa" : vqa,
            "sam" : sam_result
        })
    except:
        print(image_files[file], "Not processed")

In [ ]:
plt.imshow(sam_results[10]["image"])
show_anns(sam_results[10]["sam"])
plt.show()

In [ ]:
import pickle
with open('rad-vqa-sam.pkl', 'wb') as f:
    pickle.dump(sam_results, f)